Import the libraries needed for data handling, visualization, and neural network modeling.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , MinMaxScaler
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import accuracy_score

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


Load the customer churn dataset from the project data folder so the model can learn from real records.

In [2]:
# if it is an old .xls file
df = pd.read_csv("../../datasets/Churn_Modelling.csv")

Display the first few rows of the dataset to confirm the data loaded correctly.

In [3]:
df.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


Check the dataset shape to confirm the number of rows and columns before modeling.

In [4]:
df.shape

(10000, 14)

List the column names so the target variable and feature columns are easy to identify.

In [5]:
df.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')

Inspect the dataset types and missing values to ensure the data is ready for training.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.2 MB


Check whether any rows are duplicated so the model is not biased by repeated records.

In [7]:
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

View the class distribution of the target variable to understand churn vs non-churn balance.

In [8]:
df.duplicated().sum()

np.int64(0)

Remove columns that are not useful for prediction and keep only the relevant features.

In [9]:
df["Exited"].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

Convert categorical variables into numeric form using one-hot encoding for machine learning.

In [10]:
df =df.drop(columns = ["RowNumber" , "CustomerId" , "Surname"])
#dropping the unnecessary columns;

Split the data into training and testing sets so the model can be evaluated honestly.

In [11]:
df = pd.get_dummies(df ,columns  = ["Geography" , "Gender"] , drop_first= True)
#applying one hot encoding to categorical columns

Confirm the training and testing sets have the expected dimensions after the split.

In [12]:
X = df.drop(columns = ["Exited"])
Y = df["Exited"]
X_train , X_test , Y_train , Y_test = train_test_split(X , Y , test_size=0.2, random_state=42)

Check the output label shape to verify it matches the training data.

In [13]:
X_train.shape

(8000, 11)

Inspect the label vector to verify the target matches the number of training rows.

In [14]:
Y_train.shape

(8000,)

Standardize the input features so the neural network trains more reliably.

In [15]:
scaler = StandardScaler()#creating an scaler object
X_train_scaled = scaler.fit_transform(X_train)#scaling the input features
X_test_scaled =  scaler.transform(X_test)#scaling the output features

Create the neural network architecture with input and hidden layers for binary classification.

In [16]:
model = Sequential()#creating the model and object
model.add(Dense(4 , activation="relu" ,input_dim = 11))
#this is for creating hidden layers



c:\github_repos\ml_projects\Machine_Learning_Projects\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Add the output layer that predicts whether a customer exits or stays.

In [17]:
model.add(Dense( 1 , activation="sigmoid"))

Display the model structure to verify the layer sizes and parameter counts.

In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 4)              │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53 (212.00 B)

 Trainable params: 53 (212.00 B)

 Non-trainable params: 0 (0.00 B)

Compile the model with a binary cross-entropy loss and Adam optimizer for training.

In [19]:
model.compile(loss = "binary_crossentropy" , optimizer="Adam")
#i dont know this line

Train the neural network on the scaled training data and learn churn patterns over several epochs.

In [20]:
model.fit(X_train_scaled , Y_train , epochs=10)

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 946us/step - loss: 0.5132
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 720us/step - loss: 0.4742
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 723us/step - loss: 0.4550
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 702us/step - loss: 0.4443
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 708us/step - loss: 0.4371
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 740us/step - loss: 0.4316
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 725us/step - loss: 0.4275
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 721us/step - loss: 0.4241
Epoch 9/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 692us/step - loss: 0.4215
Epoch 10/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 684us/step - loss: 0.4186


Inspect the learned model parameters to verify the network was initialized and trained.

In [21]:
model.layers[0].get_weights()

[array([[ 0.18498239,  0.31388366,  0.099552  ,  0.06717826],
        [ 0.59992385,  0.4951157 , -1.0940832 ,  0.13755685],
        [-0.03796642, -0.05727558, -0.00429671,  0.13401091],
        [-0.2625796 , -0.04588507, -0.12653919, -0.3819682 ],
        [-0.27985418,  0.00652199, -0.00123043,  0.04058508],
        [ 0.25336355,  0.24121228,  0.18863945, -0.29170305],
        [-0.46660027,  1.001616  ,  0.0147182 ,  0.2155599 ],
        [ 0.06616684,  0.00257771,  0.22195402, -0.5148698 ],
        [ 0.28660995, -0.16972238, -0.15142918, -0.5819748 ],
        [-0.20241743, -0.4896997 , -0.31534907,  0.46762773],
        [-0.06410552, -0.31042573,  0.22051844,  0.35908633]],
       dtype=float32),
 array([0.17495124, 0.13316219, 0.7518321 , 0.20534855], dtype=float32)]

Use the trained model to predict churn probabilities for the test set.

In [22]:
Y_pred = model.predict(X_test_scaled)#predicting the output

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 829us/step


Convert the probability predictions into binary labels using a 0.5 decision threshold.

In [23]:
#now i have to make these outputs from 0 to 1 so;
#lets keep out threshold to be 0.5
Y_pred = np.where(Y_pred > 0.5 , 1 , 0)


Measure accuracy to evaluate how well the model predicts customer churn.

In [24]:
accuracy_score(Y_test , Y_pred)
# print(accuracy)

0.824

things i should do to increase accuracy of my model

#make the activation function to relu for hidden layers
#increase number of epochs 
#increase number of nodes in hidden layers
#increase number of layers
